1. Identifying Prediction Target

Our goal is to know if an airport will experience significant flights delay. At this stage the dataset does not contain a predefined prediction target for this specific goal. Therefore we are going to construct one using the variable DLY_ATC_PRE_3 (total amount of pre-departure delays caused by Air Traffic Control at an airport). Using this new continous variable we are going to generate our binary prediction target. Based on threshold that will be determined during the data exploration, that will clasify each observation as a problematic day in the airport (1), or a calm day in the airport (0).

2. Data Loading and Exploration

Since we have 5 folder each of them with at least 9 csv, one for each year. What we are going to do in order to load our data is to create a function that automatically loads each year and merges them, the only thing that changes in the name of the files inside a folder is the year, so it is an easy task. W

In [55]:
import pandas as pd
import glob
import os

def load_folder(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    df_list = []
    
    for file in files:
        df = pd.read_csv(file)
        
        # 🔹 Normalizar nombres de columnas
        df.columns = df.columns.str.strip()
        
        df_list.append(df)
    
    df = pd.concat(df_list, ignore_index=True)
    
    return df

Here we are going to merge the files inside airport_traffic into one dataframe with all the dtaa from 2016, until 2026. And we are going to repeat this process with all the other folders of the dataset. 

In [56]:
airport_traffic = load_folder("../data/airport_traffic")

In [57]:
asma_additional_time = load_folder("../data/asma_additional_time")
atc_pre_departure_delays = load_folder("../data/atc_pre_departure_delays")
ert_dly_fir = load_folder("../data/ert_dly_fir")
vertical_flight_efficiency = load_folder("../data/vertical_flight_efficiency")


Aqui deberia hacer un Exploration usando atc_pre_departure como base table y de esta manera defino ya mi prediction target. Revisar distribution y todo, revisar unas 5 variables de este dataframe.

In order to merge the datasets, first we need to see that the information is consistent in all of them, and have the same keys. If we can not handle this, we can have many duplicates, and wrong examples. We are going to start with atc_pre_departure_delays as our base table, and we are going to start merging with it. So our first step will be to observe this dataframe and find out all its details, so that we do not contaminate the dataset at the beginning.

In [79]:
print(atc_pre_departure_delays.head())
print(atc_pre_departure_delays.columns)
print(atc_pre_departure_delays.shape)

   YEAR  MONTH_NUM MONTH_MON    FLT_DATE APT_ICAO       APT_NAME STATE_NAME  \
0  2016          1       JAN  2016-01-01     EBAW        Antwerp    Belgium   
1  2016          1       JAN  2016-01-01     EBBR       Brussels    Belgium   
2  2016          1       JAN  2016-01-01     EBCI      Charleroi    Belgium   
3  2016          1       JAN  2016-01-01     EBLG          Liège    Belgium   
4  2016          1       JAN  2016-01-01     EBOS  Ostend-Bruges    Belgium   

   FLT_DEP_1  FLT_DEP_IFR_2  DLY_ATC_PRE_2  FLT_DEP_3  DLY_ATC_PRE_3  
0          4            NaN            NaN        2.0            0.0  
1        174          174.0           59.0      137.0           48.0  
2         45           45.0            0.0       43.0            0.0  
3          6            NaN            NaN        NaN            NaN  
4          7            NaN            NaN        4.0            0.0  
Index(['YEAR', 'MONTH_NUM', 'MONTH_MON', 'FLT_DATE', 'APT_ICAO', 'APT_NAME',
       'STATE_NAME', '

In [80]:
print(atc_pre_departure_delays.describe())

               YEAR     MONTH_NUM     FLT_DEP_1  FLT_DEP_IFR_2  DLY_ATC_PRE_2  \
count  1.144593e+06  1.144593e+06  1.144593e+06  215709.000000  215709.000000   
mean   2.020721e+03  6.422230e+00  6.554969e+01     138.133318      81.091725   
std    2.940016e+00  3.471963e+00  1.134604e+02     135.843501     230.923759   
min    2.016000e+03  1.000000e+00  0.000000e+00       0.000000       0.000000   
25%    2.018000e+03  3.000000e+00  5.000000e+00      41.000000       0.000000   
50%    2.021000e+03  6.000000e+00  1.800000e+01      97.000000      10.000000   
75%    2.023000e+03  9.000000e+00  7.400000e+01     197.000000      68.000000   
max    2.026000e+03  1.200000e+01  9.150000e+02     822.000000   11895.000000   

           FLT_DEP_3  DLY_ATC_PRE_3  
count  863548.000000  863548.000000  
mean       55.473795      42.593556  
std        91.619438     169.625855  
min         1.000000       0.000000  
25%         4.000000       0.000000  
50%        16.000000       0.000000  
75% 

decir que estamos haciendo

In [60]:
print(atc_pre_departure_delays["APT_ICAO"].nunique())
print(atc_pre_departure_delays["FLT_DATE"].nunique())

337
3743


decir que estamos haciendo

In [61]:
atc_pre_departure_delays.duplicated(subset=["APT_ICAO","FLT_DATE"]).sum()

0

We need to ensure consistency with the datatypes so that keys are able to work when merging.

In [62]:
atc_pre_departure_delays.dtypes

YEAR               int64
MONTH_NUM          int64
MONTH_MON         object
FLT_DATE          object
APT_ICAO          object
APT_NAME          object
STATE_NAME        object
FLT_DEP_1          int64
FLT_DEP_IFR_2    float64
DLY_ATC_PRE_2    float64
FLT_DEP_3        float64
DLY_ATC_PRE_3    float64
dtype: object

Aqui decir que quiere decir este analisis para poder empezar el merge, porque es bueno.

3. Feature Engineering

From what we saw we are going to define some provitional keys to merge. This keys may change in case any of the dataframes cannot support them. For now we are going to take year, month, flight date, and APT_ICAO (que quiere decir).*

In [63]:
keys = ["YEAR", "MONTH_NUM", "FLT_DATE", "APT_ICAO"]

Now we are going to check compatibility with airport_traffic. This should be done before merging.

In [64]:
set(atc_pre_departure_delays.columns).intersection(set(airport_traffic.columns))

{'APT_ICAO',
 'APT_NAME',
 'FLT_DATE',
 'FLT_DEP_1',
 'FLT_DEP_IFR_2',
 'MONTH_MON',
 'MONTH_NUM',
 'STATE_NAME',
 'YEAR'}

In [65]:
print(airport_traffic.head())
print(atc_pre_departure_delays.head())

   YEAR  MONTH_NUM MONTH_MON    FLT_DATE APT_ICAO    APT_NAME STATE_NAME  \
0  2016          1       JAN  2016-01-01     LATI      Tirana    Albania   
1  2016          1       JAN  2016-01-01     UDYZ     Yerevan    Armenia   
2  2016          1       JAN  2016-01-01     LOWG        Graz    Austria   
3  2016          1       JAN  2016-01-01     LOWI   Innsbruck    Austria   
4  2016          1       JAN  2016-01-01     LOWK  Klagenfurt    Austria   

   FLT_DEP_1  FLT_ARR_1  FLT_TOT_1  FLT_DEP_IFR_2  FLT_ARR_IFR_2  \
0         24         27         51            NaN            NaN   
1          8         15         23            NaN            NaN   
2          6          7         13            NaN            NaN   
3         26         32         58            NaN            NaN   
4          3          4          7            NaN            NaN   

   FLT_TOT_IFR_2  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4            NaN  
   YEAR  MONTH_NUM M

In [66]:
print(atc_pre_departure_delays.groupby(["APT_ICAO","FLT_DATE"]).size().head())
print(airport_traffic.groupby(["APT_ICAO","FLT_DATE"]).size().head())

APT_ICAO  FLT_DATE  
BIKF      2023-12-01    1
          2023-12-02    1
          2023-12-03    1
          2023-12-04    1
          2023-12-05    1
dtype: int64
APT_ICAO  FLT_DATE  
BIKF      2024-01-01    1
          2024-01-02    1
          2024-01-03    1
          2024-01-04    1
          2024-01-05    1
dtype: int64


From here we can observe that the keys we assigned provisionally work for airport_traffic. Since we can make each row an airport and date. 

In [67]:
merged_data = atc_pre_departure_delays.merge(
    airport_traffic,
    on=["YEAR","MONTH_NUM","FLT_DATE","APT_ICAO"],
    how="left"
)

In [69]:
merged_data.head()

,YEAR,MONTH_NUM,MONTH_MON_x,FLT_DATE,APT_ICAO,APT_NAME_x,STATE_NAME_x,FLT_DEP_1_x,FLT_DEP_IFR_2_x,DLY_ATC_PRE_2,...,DLY_ATC_PRE_3,MONTH_MON_y,APT_NAME_y,STATE_NAME_y,FLT_DEP_1_y,FLT_ARR_1,FLT_TOT_1,FLT_DEP_IFR_2_y,FLT_ARR_IFR_2,FLT_TOT_IFR_2
0,2016,1,JAN,2016-01-01,EBAW,Antwerp,Belgium,4,NaN,NaN,...,0.0,JAN,Antwerp,Belgium,4.0,3.0,7.0,NaN,NaN,NaN
1,2016,1,JAN,2016-01-01,EBBR,Brussels,Belgium,174,174.0,59.0,...,48.0,JAN,Brussels,Belgium,174.0,171.0,345.0,174.0,161.0,335.0
2,2016,1,JAN,2016-01-01,EBCI,Charleroi,Belgium,45,45.0,0.0,...,0.0,JAN,Charleroi,Belgium,45.0,47.0,92.0,45.0,45.0,90.0
3,2016,1,JAN,2016-01-01,EBLG,Liège,Belgium,6,NaN,NaN,...,NaN,JAN,Liège,Belgium,6.0,7.0,13.0,NaN,NaN,NaN
4,2016,1,JAN,2016-01-01,EBOS,Ostend-Bruges,Belgium,7,NaN,NaN,...,0.0,JAN,Ostend-Bruges,Belgium,7.0,7.0,14.0,NaN,NaN,NaN


In [70]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144594 entries, 0 to 1144593
Data columns (total 21 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   YEAR             1144594 non-null  int64  
 1   MONTH_NUM        1144594 non-null  int64  
 2   MONTH_MON_x      1144594 non-null  object 
 3   FLT_DATE         1144594 non-null  object 
 4   APT_ICAO         1144594 non-null  object 
 5   APT_NAME_x       1144594 non-null  object 
 6   STATE_NAME_x     1144594 non-null  object 
 7   FLT_DEP_1_x      1144594 non-null  int64  
 8   FLT_DEP_IFR_2_x  215709 non-null   float64
 9   DLY_ATC_PRE_2    215709 non-null   float64
 10  FLT_DEP_3        863549 non-null   float64
 11  DLY_ATC_PRE_3    863549 non-null   float64
 12  MONTH_MON_y      1126093 non-null  object 
 13  APT_NAME_y       1126093 non-null  object 
 14  STATE_NAME_y     1126093 non-null  object 
 15  FLT_DEP_1_y      1126093 non-null  float64
 16  FLT_ARR_1        1

Here we can observe everything worked well in the merging. But we can also observe that now we have duplicate variables, so to work with integrity what we are going to do is drop the duplicate columns. That are: "MONTH_MON_y", "APT_NAME_y", "STATE_NAME_y", "FLT_DEP_1_y", "FLT_DEP_IFR_2_y". And we can also observe that this wasnt a one to one join, since there where some days that any of the two tables didn't had a record, this could bring bias if its not well treated.

In [71]:
merged_data = merged_data.drop(columns=[
    "MONTH_MON_y",
    "APT_NAME_y",
    "STATE_NAME_y",
    "FLT_DEP_1_y",
    "FLT_DEP_IFR_2_y"
], errors="ignore")

We are going to rename the columns that where duplicated to the ones we had before, since pandas changed its name to _x and _y; because it didn't know which was the main one. 

In [72]:
merged_data = merged_data.rename(columns={
    "MONTH_MON_x": "MONTH_MON",
    "APT_NAME_x": "APT_NAME",
    "STATE_NAME_x": "STATE_NAME",
    "FLT_DEP_1_x": "FLT_DEP_1",
    "FLT_DEP_IFR_2_x": "FLT_DEP_IFR_2"
})

In [ ]:
merged_data.isnull().mean().sort_values(ascending=False)

FLT_DEP_IFR_2    0.811541
DLY_ATC_PRE_2    0.811541
FLT_DEP_3        0.245541
DLY_ATC_PRE_3    0.245541
YEAR             0.000000
MONTH_NUM        0.000000
MONTH_MON        0.000000
FLT_DATE         0.000000
APT_ICAO         0.000000
APT_NAME         0.000000
STATE_NAME       0.000000
FLT_DEP_1        0.000000
dtype: float64

Now we are going to try to merge with asma_additional_time, so we are going to do a similar analysis to the one we did with airport_traffic. So that we know if they keys, and information is consistent to merge.

In [76]:
print(asma_additional_time.columns)
print(asma_additional_time.head())

Index(['YEAR', 'MONTH_NUM', 'MONTH_MON', 'APT_ICAO', 'APT_NAME', 'STATE_NAME',
       'TF', 'VALID_FL', 'NO_REF', 'TOTAL_REF_NB_FL', 'TOTAL_REF_TIME_MIN',
       'TOTAL_ADD_TIME_MIN', 'COMMENT'],
      dtype='object')
   YEAR  MONTH_NUM MONTH_MON APT_ICAO              APT_NAME STATE_NAME  \
0  2018          1       JAN     LOWW                Vienna    Austria   
1  2018          1       JAN     EBBR              Brussels    Belgium   
2  2018          1       JAN     EBCI  Brussels - Charleroi    Belgium   
3  2018          1       JAN     LBSF                 Sofia   Bulgaria   
4  2018          1       JAN     LDZA                Zagreb    Croatia   

       TF  VALID_FL  NO_REF  TOTAL_REF_NB_FL  TOTAL_REF_TIME_MIN  \
0  8514.0    7449.0    49.0           7400.0         80204.07417   
1  8528.0    6877.0   110.0           6767.0         74294.40667   
2  2195.0    1516.0    32.0           1484.0         15732.61583   
3  2229.0    1750.0    10.0           1740.0         18554.51917 

We can see that this dataset does not has FLT_DATE, meaning that the data is not daily, it is by month. So what we can do is generalize this data, and use it as context features. In this step we need to take a decision between converting our project into a daily prediction, or monthly prediction. Since our goal is to predict the next day, we need to stick with daily predctions. So we can rather not use asma_additional_time, or use it as monthly context feature merged at airport-month level. In this case, I belive this is going to be the best decision. Because this features may be very usefull, and we can add them right now, and if they don't work well, we can drop them later on. 

Dice que escribamos algo asi: 
We validate the dataset granularity by checking whether observations are already aggregated at the airport-day level. This ensures that no unintended duplication occurs during the merging process.